# 💧 AquaSense AI — 07: Explanation Consistency Analysis
**Project:** Intelligent Water Quality Assessment and Potability Prediction Using Explainable Machine Learning  

### Overview:
In this notebook, we perform the **cross-method XAI consistency analysis** comparing:
1. **SHAP** (Game-theoretic Shapley values)
2. **LIME** (Local sparse linear surrogates)
3. **Permutation Importance** (Empirical metric decay under perturbation)

We compute:
- Spearman rank correlation matrix
- Agreement classification table (Strong / Moderate / Weak)
- Normalized side-by-side attribution charts
- Plain English trustworthiness conclusions


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import DataLoader
from src.data.preprocessor import WaterQualityPreprocessor
from src.models.trainer import ModelTrainer
from src.explainability.consistency import ExplanationConsistencyAnalyzer
from src.utils.visualization import set_plot_style

set_plot_style()
print("Consistency analysis modules loaded.")


## 1. Load Data, Preprocessor & Best Model


In [ ]:
dl = DataLoader(data_path="../data/raw/water_potability.csv")
df = dl.load()
X_train, X_test, y_train, y_test = dl.split(df, test_size=0.20, random_state=42)

preprocessor = WaterQualityPreprocessor.load("../models/preprocessor.pkl")
X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)

best_model = ModelTrainer.load_single("best_model", path="../models")
print("Model loaded successfully.")


## 2. Compute Rankings Across All 3 Interpretability Methods


In [ ]:
analyzer = ExplanationConsistencyAnalyzer(random_state=42)
rankings, scores = analyzer.compute_all_rankings(
    best_model, X_train_proc, X_test_proc.head(50), y_test.head(50),
    n_lime_samples=15, n_perm_repeats=10
)

df_rankings = pd.DataFrame(rankings)
df_rankings.index = range(1, len(df_rankings) + 1)
df_rankings.head(10)


## 3. Spearman Rank Correlation Matrix


In [ ]:
corr_matrix = analyzer.spearman_correlation_matrix(rankings)
print("Spearman Rank Correlation Matrix:")
display(corr_matrix)

heat_fig = analyzer.plot_correlation_heatmap(corr_matrix)
plt.show()


## 4. Feature Agreement Classification Table


In [ ]:
agreement_df = analyzer.agreement_table(rankings)
display(agreement_df)


## 5. Normalized Consistency Comparison Bar Chart


In [ ]:
comp_fig = analyzer.plot_consistency_comparison(rankings, scores)
plt.show()


## 6. Plain English Interpretability & Trustworthiness Report


In [ ]:
report = analyzer.generate_consistency_report(rankings, corr_matrix, agreement_df)
print(report)
